# 使用Meta家族模型进行构建

## 介绍

本课将涵盖：

- 探索两个主要的Meta家族模型 - Llama 3.1和Llama 3.2
- 了解每个模型的使用场景和应用
- 通过代码示例展示每个模型的独特功能


## Meta家族模型

在本课中，我们将探索Meta家族或“Llama Herd”的2个模型——Llama 3.1和Llama 3.2

这些模型有不同的变体，并且可以在[Microsoft Foundry Models catalog](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst)中获取。

> **注意：** GitHub模型将在2026年7月底退休。这里有更多关于如何使用[Microsoft Foundry Models](https://learn.microsoft.com/en-us/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst)进行AI模型原型设计的详细信息。

模型变体：
- Llama 3.1 - 70B Instruct
- Llama 3.1 - 405B Instruct
- Llama 3.2 - 11B Vision Instruct
- Llama 3.2 - 90B Vision Instruct

*备注：Llama 3也在Microsoft Foundry Models中提供，但本课不涉及*



## Llama 3.1 

在4050亿参数规模下，Llama 3.1 属于开源大型语言模型（LLM）类别。 

该模型是对早期版本 Llama 3 的升级，提供了： 

- 更大的上下文窗口 - 128k 令牌，较之前的 8k 令牌 
- 更大的最大输出令牌数 - 4096，较之前的 2048 
- 更好的多语言支持 - 由于训练令牌数量增加 

这些使得 Llama 3.1 能够处理更复杂的生成式人工智能应用场景，包括： 
- 原生函数调用 - 能够调用 LLM 工作流外部的工具和函数
- 更好的 RAG 性能 - 归功于更大的上下文窗口 
- 合成数据生成 - 能够为微调等任务创建有效数据 



### 原生函数调用 

Llama 3.1 已经过微调，以更有效地进行函数或工具调用。它还内置了两个模型可以根据用户提示识别并调用的工具。这些工具是： 

- **Brave Search** - 可用于通过网络搜索获取最新信息，比如天气 
- **Wolfram Alpha** - 可用于更复杂的数学计算，因此不需要自己编写函数。 

你也可以创建自己的自定义工具，供大语言模型调用。 

下面的代码示例中： 

- 我们在系统提示中定义了可用工具（brave_search，wolfram_alpha）。 
- 发送一个用户提示，询问某个城市的天气。 
- 大语言模型将通过调用 Brave Search 工具响应，看起来像这样 `<|python_tag|>brave_search.call(query="Stockholm weather")` 

*注意：此示例仅进行工具调用，如果你想获得结果，需要在 Brave API 页面创建一个免费账户并定义该函数本身* 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Meta-Llama-3.1-405B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

尽管 Llama 3.1 是一个大型语言模型，但它的一个限制是多模态能力。也就是说，能够使用不同类型的输入，例如图像作为提示并提供响应。这种能力是 Llama 3.2 的主要特性之一。这些特性还包括： 

- 多模态 —— 具有同时处理文本和图像提示的能力 
- 小到中等规模的变体（11B 和 90B）—— 提供灵活的部署选项， 
- 仅文本变体（1B 和 3B） —— 允许模型部署在边缘/移动设备上并提供低延迟 

多模态支持代表了开源模型领域的一个重大进步。下面的代码示例同时接受图像和文本提示，以获得 Llama 3.2 90B 对图像的分析。 

### Llama 3.2 的多模态支持


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Llama-3.2-90B-Vision-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## 学习不会止步于此，继续您的旅程

完成本课后，请查看我们的[生成式 AI 学习合集](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst)，继续提升您的生成式 AI 知识！


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
